# Workshop 2 - Machine Learning y Deep Learning Aplicado

Este notebook deja una base completa para desarrollar los dos problemas del workshop:

1. Clasificacion: deteccion de fatiga muscular con señales EMG.
2. Regresion: estimacion de edad a partir de imagenes faciales.

> Nota: la estructura ya incluye celdas de analisis, preprocesamiento, entrenamiento y evaluacion para que solo completes o ajustes segun tus resultados.

## Ruta de trabajo del notebook

### Problema 1 - Clasificacion (EMG)
1. Importacion de datos y analisis preliminar.
2. Construccion del nuevo dataset por ventanas de 1 segundo.
3. EDA e interpretacion.
4. Preprocesamiento y particion train/val/test.
5. Entrenamiento de modelos clasicos con GridSearchCV.
6. Entrenamiento DNN (Keras) con regularizacion y busqueda de hiperparametros.
7. Comparacion de modelos, curvas train/val y evaluacion final del mejor modelo.
8. Prueba con muestra artificial.

### Problema 2 - Regresion (imagenes)
1. Carga y analisis preliminar del dataset de rostros.
2. EDA de edades e inspeccion visual.
3. Baseline reproducible con sklearn.
4. CNN de regresion (Keras) con dropout, batch norm y early stopping.
5. Metricas MAE, RMSE y R2 en train/val/test + prueba con imagen modificada.

In [1]:
import os
import random
import warnings
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sns = None
try:
    sns = importlib.import_module("seaborn")
    HAS_SNS = True
except Exception:
    HAS_SNS = False

tf = None
keras = None
try:
    tf = importlib.import_module("tensorflow")
    keras = tf.keras
    HAS_TF = True
except Exception:
    HAS_TF = False

from scipy.signal import welch

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    median_absolute_error,
    explained_variance_score,
    max_error
    )
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, RandomForestRegressor, GradientBoostingRegressor

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
if HAS_TF:
    tf.random.set_seed(SEED)

warnings.filterwarnings("ignore")
if HAS_SNS:
    sns.set_theme(style="whitegrid")

print("Entorno listo con herramientas vistas en el curso (pandas/numpy/sklearn + Keras opcional).")
if not HAS_SNS:
    print("Seaborn no disponible: se usaran graficos base de matplotlib.")
if not HAS_TF:
    print("TensorFlow/Keras no disponible: las celdas DNN/CNN se saltaran automaticamente.")

Entorno listo con herramientas vistas en el curso (pandas/numpy/sklearn + Keras opcional).
Seaborn no disponible: se usaran graficos base de matplotlib.
TensorFlow/Keras no disponible: las celdas DNN/CNN se saltaran automaticamente.


## Problema 1 - Clasificacion: Fatiga muscular en ciclismo (EMG)

### 1) Importacion y analisis preliminar
En esta seccion se realiza la importacion del dataset siguiendo la logica de TutorialNewDataSetCreation y se prepara el target binario (0 normal, 1 desgaste).

In [ ]:
# Importacion del dataset EMG tal como el tutorial
# (lectura con pandas, fs=1000 Hz, ventanas de 1 segundo y canales definidos)

dataset_path = "muscle_fatigue_cycling.csv"
if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        f"No se encontro {dataset_path}. Ubicalo en la raiz del workspace o ajusta esta ruta."
    )

df = pd.read_csv(dataset_path)

fs = 1000
window_size = fs

channels = [
    "Muscle_1",
    "Muscle_2",
    "Muscle_3",
    "Muscle_4",
    "Muscle_5",
    "Muscle_6",
    "Muscle_7",
    "Muscle_8",
]

if "Target" not in df.columns:
    raise ValueError("El dataset debe tener una columna Target.")

# Ajuste de etiquetas: 0 normal, 1 desgaste (2 pasa a 1)
df["Target"] = df["Target"].replace({2: 1}).astype(int)

n_windows = len(df) // window_size

print(f"Total de muestras: {len(df)}")
print(f"Ventanas de 1 segundo: {n_windows}")
print(f"Columnas disponibles: {list(df.columns)}")

display(df.head())

### 2) Feature engineering por ventanas de 1 segundo
Se calculan caracteristicas de dominio del tiempo y de frecuencia para cada canal, sin usar la columna Time para evitar fuga de informacion temporal.

In [ ]:
def extraer_caracteristicas(ventana, fs=1000):
    """Extrae caracteristicas de tiempo y frecuencia de una ventana 1D."""
    rms = np.sqrt(np.mean(ventana**2))
    varianza = np.var(ventana)
    zcr = np.sum(np.diff(np.sign(ventana)) != 0)
    mav = np.mean(np.abs(ventana))

    freqs, psd = welch(ventana, fs=fs)
    pot_total = np.sum(psd)
    frec_media = np.sum(freqs * psd) / (np.sum(psd) + 1e-12)
    pot_acum = np.cumsum(psd)
    idx_median = np.searchsorted(pot_acum, pot_acum[-1] / 2)
    frec_mediana = freqs[min(idx_median, len(freqs) - 1)]

    return [rms, varianza, zcr, mav, pot_total, frec_media, frec_mediana]


nombres_feats = ["rms", "var", "zcr", "mav", "pot", "f_media", "f_mediana"]
filas = []

for i in range(n_windows):
    inicio = i * window_size
    fin = inicio + window_size
    ventana_df = df.iloc[inicio:fin]

    fila = {}
    for canal in channels:
        if canal not in ventana_df.columns:
            raise ValueError(f"No existe el canal {canal} en el dataset.")

        ventana = ventana_df[canal].values
        feats = extraer_caracteristicas(ventana, fs=fs)

        for nombre, valor in zip(nombres_feats, feats):
            fila[f"{canal}_{nombre}"] = valor

    target_mode = ventana_df["Target"].mode()
    fila["target"] = int(target_mode.iloc[0]) if not target_mode.empty else int(ventana_df["Target"].iloc[-1])

    filas.append(fila)

nuevo_df = pd.DataFrame(filas)
nuevo_df.to_csv("emg_features_1s.csv", index=False)

print(f"Shape nuevo dataset: {nuevo_df.shape}")
print("Archivo guardado: emg_features_1s.csv")
display(nuevo_df.head())

In [ ]:
# EDA minimo inicial solicitado

if "Time" in df.columns:
    eje_x = df["Time"].iloc[:5000]
    etiqueta_x = "Time (s)"
else:
    eje_x = np.arange(min(len(df), 5000)) / fs
    etiqueta_x = "Tiempo aproximado (s)"

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for i, canal in enumerate(channels[:4]):
    ax = axes[i // 2, i % 2]
    ax.plot(eje_x, df[canal].iloc[: len(eje_x)], linewidth=0.9)
    ax.set_title(f"Senal {canal} (primeras {len(eje_x)} muestras)")
    ax.set_xlabel(etiqueta_x)
    ax.set_ylabel("Amplitud")
plt.tight_layout()
plt.show()

print("Balance de clases en el dataset original:")
display(df["Target"].value_counts(normalize=True).rename("proporcion"))

print("Resumen estadistico del dataset de caracteristicas:")
display(nuevo_df.describe().T)

conteo_target = nuevo_df["target"].value_counts().sort_index()
plt.figure(figsize=(6, 4))
if HAS_SNS:
    sns.countplot(data=nuevo_df, x="target", hue="target", legend=False, palette="Set2")
else:
    plt.bar(conteo_target.index.astype(str), conteo_target.values, color=["#66c2a5", "#fc8d62"])
plt.title("Balance de clases en nuevo dataset")
plt.xlabel("target")
plt.ylabel("conteo")
plt.show()

plt.figure(figsize=(12, 8))
corr = nuevo_df.corr(numeric_only=True)
if HAS_SNS:
    sns.heatmap(corr, cmap="coolwarm", center=0)
else:
    plt.imshow(corr.values, cmap="coolwarm", aspect="auto")
    plt.colorbar()
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.index)), corr.index)
plt.title("Mapa de correlaciones (nuevo dataset)")
plt.tight_layout()
plt.show()

# Particion 70/15/15
X = nuevo_df.drop(columns=["target"])
y = nuevo_df["target"].astype(int)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.1765,
    random_state=SEED,
    stratify=y_train_full,
)

print("Shapes:")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")

preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)


def metricas_clasificacion(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

In [ ]:
# Entrenamiento de modelos clasicos con ajuste de hiperparametros (GridSearchCV)

espacios = {
    "kNN": (
        KNeighborsClassifier(),
        {
            "modelo__n_neighbors": [3, 5, 7, 9, 11],
            "modelo__weights": ["uniform", "distance"],
        },
    ),
    "DecisionTree": (
        DecisionTreeClassifier(random_state=SEED),
        {
            "modelo__max_depth": [3, 5, 8, 12, None],
            "modelo__min_samples_split": [2, 5, 10],
            "modelo__min_samples_leaf": [1, 2, 4],
        },
    ),
    "RandomForest": (
        RandomForestClassifier(random_state=SEED),
        {
            "modelo__n_estimators": [100, 200],
            "modelo__max_depth": [None, 8, 12],
            "modelo__min_samples_split": [2, 5],
            "modelo__min_samples_leaf": [1, 2],
        },
    ),
    "GradientBoosting": (
        GradientBoostingClassifier(random_state=SEED),
        {
            "modelo__n_estimators": [100, 200],
            "modelo__learning_rate": [0.01, 0.05, 0.1],
            "modelo__max_depth": [2, 3],
            "modelo__subsample": [0.8, 1.0],
        },
    ),
}

resultados = []
mejores_modelos = {}
mejores_parametros_clf = {}

for nombre, (estimador, params) in espacios.items():
    print(f"\nEntrenando {nombre}...")

    pipe = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("modelo", estimador),
        ]
    )

    busqueda = GridSearchCV(
        estimator=pipe,
        param_grid=params,
        scoring="f1",
        cv=3,
        n_jobs=-1,
        verbose=0,
    )

    busqueda.fit(X_train, y_train)
    mejor = busqueda.best_estimator_
    mejores_modelos[nombre] = mejor
    mejores_parametros_clf[nombre] = busqueda.best_params_

    for split_name, X_split, y_split in [
        ("train", X_train, y_train),
        ("val", X_val, y_val),
        ("test", X_test, y_test),
    ]:
        pred = mejor.predict(X_split)
        fila = {"modelo": nombre, "split": split_name, **metricas_clasificacion(y_split, pred)}
        resultados.append(fila)

    print(f"Mejores hiperparametros ({nombre}): {busqueda.best_params_}")

resultados_clf = pd.DataFrame(resultados)
display(resultados_clf.sort_values(by=["modelo", "split"]).reset_index(drop=True))

# Curva train-val-test (F1) para detectar brechas de generalizacion
plt.figure(figsize=(8, 5))
orden_split = ["train", "val", "test"]
for nombre in resultados_clf["modelo"].unique():
    curva = resultados_clf[resultados_clf["modelo"] == nombre].set_index("split").reindex(orden_split)
    plt.plot(orden_split, curva["f1"].values, marker="o", linewidth=2, label=nombre)
plt.title("F1 por split en modelos clasicos")
plt.xlabel("Split")
plt.ylabel("F1")
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

ranking_val = resultados_clf[resultados_clf["split"] == "val"].sort_values("f1", ascending=False)
mejor_modelo_clasico_nombre = ranking_val.iloc[0]["modelo"]
print(f"Mejor modelo clasico segun validacion (F1): {mejor_modelo_clasico_nombre}")

In [ ]:
# Modelo DNN para clasificacion (Keras) con 3 capas ocultas y regularizacion

dnn_disponible = False
dnn_best_model = None
dnn_best_history = None
dnn_best_config = None
preprocess_dnn = None

if not HAS_TF:
    print("TensorFlow/Keras no disponible. Se omite la seccion DNN.")
else:
    dnn_disponible = True

    preprocess_dnn = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    X_train_dnn = preprocess_dnn.fit_transform(X_train).astype("float32")
    X_val_dnn = preprocess_dnn.transform(X_val).astype("float32")
    X_test_dnn = preprocess_dnn.transform(X_test).astype("float32")

    y_train_np = y_train.values.astype("float32")
    y_val_np = y_val.values.astype("float32")
    y_test_np = y_test.values.astype("float32")

    def crear_dnn(input_dim, unidades=(128, 64, 32), dropout=0.2, l2_reg=1e-4, lr=1e-3):
        reg = keras.regularizers.l2(l2_reg)
        modelo = keras.Sequential(
            [
                keras.layers.Input(shape=(input_dim,)),
                keras.layers.BatchNormalization(),
                keras.layers.Dense(unidades[0], activation="relu", kernel_regularizer=reg),
                keras.layers.Dropout(dropout),
                keras.layers.Dense(unidades[1], activation="relu", kernel_regularizer=reg),
                keras.layers.Dropout(dropout),
                keras.layers.Dense(unidades[2], activation="relu", kernel_regularizer=reg),
                keras.layers.Dense(1, activation="sigmoid"),
            ]
        )
        modelo.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss="binary_crossentropy",
            metrics=[
                keras.metrics.BinaryAccuracy(name="accuracy"),
                keras.metrics.Precision(name="precision"),
                keras.metrics.Recall(name="recall"),
            ],
        )
        return modelo

    espacio_dnn = [
        {"unidades": (128, 64, 32), "dropout": 0.20, "l2_reg": 1e-4, "lr": 1e-3, "batch_size": 32},
        {"unidades": (64, 32, 16), "dropout": 0.30, "l2_reg": 1e-4, "lr": 1e-3, "batch_size": 32},
        {"unidades": (128, 64, 32), "dropout": 0.30, "l2_reg": 1e-3, "lr": 5e-4, "batch_size": 32},
        {"unidades": (256, 128, 64), "dropout": 0.20, "l2_reg": 1e-4, "lr": 5e-4, "batch_size": 64},
    ]

    dnn_resultados = []
    mejor_val_f1 = -1.0

    for i, cfg in enumerate(espacio_dnn, start=1):
        print(f"\nEntrenando DNN config {i}/{len(espacio_dnn)}: {cfg}")
        modelo = crear_dnn(
            input_dim=X_train_dnn.shape[1],
            unidades=cfg["unidades"],
            dropout=cfg["dropout"],
            l2_reg=cfg["l2_reg"],
            lr=cfg["lr"],
        )

        callback_es = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=8, restore_best_weights=True
        )

        hist = modelo.fit(
            X_train_dnn,
            y_train_np,
            validation_data=(X_val_dnn, y_val_np),
            epochs=60,
            batch_size=cfg["batch_size"],
            callbacks=[callback_es],
            verbose=0,
        )

        pred_val = (modelo.predict(X_val_dnn, verbose=0).ravel() >= 0.5).astype(int)
        f1_val = f1_score(y_val, pred_val, zero_division=0)
        dnn_resultados.append(
            {
                "config": str(cfg),
                "f1_val": f1_val,
                "epochs": len(hist.history["loss"]),
            }
        )

        if f1_val > mejor_val_f1:
            mejor_val_f1 = f1_val
            dnn_best_model = modelo
            dnn_best_history = hist
            dnn_best_config = cfg

    display(pd.DataFrame(dnn_resultados).sort_values("f1_val", ascending=False).reset_index(drop=True))
    print(f"Mejor configuracion DNN: {dnn_best_config}")

    # Evitar duplicados si la celda se ejecuta varias veces
    resultados_clf = resultados_clf[resultados_clf["modelo"] != "DNN_Keras"].copy()

    for split_name, X_split, y_split in [
        ("train", X_train_dnn, y_train),
        ("val", X_val_dnn, y_val),
        ("test", X_test_dnn, y_test),
    ]:
        pred = (dnn_best_model.predict(X_split, verbose=0).ravel() >= 0.5).astype(int)
        fila = {"modelo": "DNN_Keras", "split": split_name, **metricas_clasificacion(y_split, pred)}
        resultados_clf = pd.concat([resultados_clf, pd.DataFrame([fila])], ignore_index=True)

    # Curvas de entrenamiento y validacion del mejor DNN
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(dnn_best_history.history["loss"], label="train_loss")
    plt.plot(dnn_best_history.history["val_loss"], label="val_loss")
    plt.title("DNN - Curva de perdida")
    plt.xlabel("Epoca")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(dnn_best_history.history["accuracy"], label="train_acc")
    plt.plot(dnn_best_history.history["val_accuracy"], label="val_acc")
    plt.title("DNN - Curva de accuracy")
    plt.xlabel("Epoca")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Curva comparativa train/val/test para todos los modelos
    plt.figure(figsize=(8, 5))
    orden_split = ["train", "val", "test"]
    for nombre in resultados_clf["modelo"].unique():
        curva = resultados_clf[resultados_clf["modelo"] == nombre].set_index("split").reindex(orden_split)
        plt.plot(orden_split, curva["f1"].values, marker="o", linewidth=2, label=nombre)
    plt.title("F1 por split (clasicos + DNN)")
    plt.xlabel("Split")
    plt.ylabel("F1")
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    display(resultados_clf.sort_values(by=["modelo", "split"]).reset_index(drop=True))

In [ ]:
# Evaluacion final unificada del mejor clasificador (clasicos + DNN)

ranking_general = resultados_clf[resultados_clf["split"] == "val"].sort_values("f1", ascending=False)
mejor_modelo_global = ranking_general.iloc[0]["modelo"]
print(f"Mejor clasificador segun validacion (F1): {mejor_modelo_global}")

X_train_val = pd.concat([X_train, X_val], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

if mejor_modelo_global == "DNN_Keras":
    if not dnn_disponible:
        raise RuntimeError("Se selecciono DNN pero TensorFlow no esta disponible.")

    X_train_val_dnn = preprocess_dnn.fit_transform(X_train_val).astype("float32")
    X_test_dnn_final = preprocess_dnn.transform(X_test).astype("float32")
    y_train_val_np = y_train_val.values.astype("float32")

    modelo_final_clf = crear_dnn(
        input_dim=X_train_val_dnn.shape[1],
        unidades=dnn_best_config["unidades"],
        dropout=dnn_best_config["dropout"],
        l2_reg=dnn_best_config["l2_reg"],
        lr=dnn_best_config["lr"],
    )

    callback_es = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True
    )

    _ = modelo_final_clf.fit(
        X_train_val_dnn,
        y_train_val_np,
        validation_split=0.1,
        epochs=60,
        batch_size=dnn_best_config["batch_size"],
        callbacks=[callback_es],
        verbose=0,
    )

    y_test_pred_prob = modelo_final_clf.predict(X_test_dnn_final, verbose=0).ravel()
    y_test_pred = (y_test_pred_prob >= 0.5).astype(int)
else:
    modelo_final_clf = mejores_modelos[mejor_modelo_global]
    modelo_final_clf.fit(X_train_val, y_train_val)
    y_test_pred = modelo_final_clf.predict(X_test)
    y_test_pred_prob = None

print("\nReporte final en X_test:")
print(classification_report(y_test, y_test_pred, digits=4))

cm = confusion_matrix(y_test, y_test_pred)
cm_df = pd.DataFrame(cm, index=["real_0", "real_1"], columns=["pred_0", "pred_1"])
display(cm_df)

plt.figure(figsize=(5, 4))
if HAS_SNS:
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
else:
    plt.imshow(cm_df.values, cmap="Blues", aspect="auto")
    for i in range(cm_df.shape[0]):
        for j in range(cm_df.shape[1]):
            plt.text(j, i, str(cm_df.values[i, j]), ha="center", va="center", color="black")
    plt.xticks(range(cm_df.shape[1]), cm_df.columns)
    plt.yticks(range(cm_df.shape[0]), cm_df.index)
plt.title(f"Matriz de confusion final - {mejor_modelo_global}")
plt.xlabel("Prediccion")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

# Boxplots de caracteristicas representativas por clase predicha
df_eval = X_test.copy()
df_eval["pred_clase"] = y_test_pred
features_rms = [c for c in X_test.columns if c.endswith("_rms")][:2]
features_freq = [c for c in X_test.columns if c.endswith("_f_mediana")][:2]
features_box = features_rms + features_freq
if not features_box:
    features_box = list(X_test.columns[:4])

for feat in features_box:
    plt.figure(figsize=(6, 4))
    if HAS_SNS:
        ax = sns.boxplot(data=df_eval, x="pred_clase", y=feat, hue="pred_clase", palette="Set2")
        if ax.get_legend() is not None:
            ax.get_legend().remove()
    else:
        valores_0 = df_eval[df_eval["pred_clase"] == 0][feat].values
        valores_1 = df_eval[df_eval["pred_clase"] == 1][feat].values
        plt.boxplot([valores_0, valores_1], labels=["0", "1"])
    plt.title(f"{feat} por clase predicha")
    plt.xlabel("Clase predicha")
    plt.ylabel(feat)
    plt.tight_layout()
    plt.show()

# Prueba con muestra artificial aproximada a datos reales
muestra_artificial = X_train_val.sample(1, random_state=SEED).copy()
ruido = np.random.normal(loc=0.0, scale=0.03, size=muestra_artificial.shape)
muestra_artificial.iloc[0, :] = muestra_artificial.iloc[0, :] * (1 + ruido[0])

if mejor_modelo_global == "DNN_Keras":
    muestra_artificial_dnn = preprocess_dnn.transform(muestra_artificial).astype("float32")
    pred_artificial = int((modelo_final_clf.predict(muestra_artificial_dnn, verbose=0).ravel()[0] >= 0.5))
else:
    pred_artificial = int(modelo_final_clf.predict(muestra_artificial)[0])

print(f"Prediccion muestra artificial (0=normal, 1=fatiga): {pred_artificial}")

metricas_finales = metricas_clasificacion(y_test, y_test_pred)
print("\nResumen rapido:")
print(metricas_finales)
if metricas_finales["f1"] >= 0.85:
    print("Interpretacion: clasificador fuerte para este conjunto de test.")
elif metricas_finales["f1"] >= 0.70:
    print("Interpretacion: clasificador aceptable, con margen de mejora.")
else:
    print("Interpretacion: clasificador limitado; revisar features y ajuste de hiperparametros.")

## Nota de alcance y trazabilidad

Este notebook se resolvio con herramientas vistas en Curso IA hasta el 17 de marzo:
- pandas, numpy, matplotlib y seaborn para EDA.
- scikit-learn para pipeline, particion y GridSearchCV.
- tensorflow/keras (Lecture08) para DNN y CNN.

Si TensorFlow no esta instalado en tu entorno, las celdas deep learning se saltan automaticamente y el flujo clasico sigue funcionando.

## Problema 2 - Regresion: Estimacion de edad desde imagenes

Esta parte incluye dos enfoques complementarios, ambos vistos en clase:
- baseline tabular con sklearn (imagenes vectorizadas),
- CNN de regresion con Keras,
- comparacion de metricas en train/val/test,
- prueba con muestra artificial y cambios visuales.

In [ ]:
# 1) Ubicar dataset de imagenes

faces_root_candidates = [
    Path("faces"),
    Path("UTKFace"),
    Path("regresion") / "faces",
    Path("regresion") / "UTKFace",
]

faces_root = None
for p in faces_root_candidates:
    if p.exists() and p.is_dir():
        faces_root = p
        break

if faces_root is None:
    print("No se encontro carpeta de imagenes de rostros en rutas comunes.")
    print("Crea una carpeta 'faces' o 'UTKFace' en la raiz del workspace y vuelve a ejecutar.")
else:
    print(f"Dataset de imagenes encontrado en: {faces_root}")

In [ ]:
# 2) Construir metadata (path, age)

if faces_root is not None:
    extensiones = ["*.jpg", "*.jpeg", "*.png"]
    rutas = []
    for ext in extensiones:
        rutas.extend(list(faces_root.rglob(ext)))

    registros = []
    for ruta in rutas:
        nombre = ruta.stem
        partes = nombre.split("_")
        if len(partes) >= 1 and partes[0].isdigit():
            edad = int(partes[0])
            registros.append({"path": str(ruta), "age": edad})

    meta_faces = pd.DataFrame(registros)
    print(f"Total imagenes con etiqueta de edad valida: {len(meta_faces)}")
    display(meta_faces.head())
else:
    meta_faces = pd.DataFrame(columns=["path", "age"])

In [ ]:
# 3) EDA inicial del target edad

if not meta_faces.empty:
    print("Resumen descriptivo de edad:")
    display(meta_faces["age"].describe())

    plt.figure(figsize=(10, 4))
    if HAS_SNS:
        sns.histplot(meta_faces["age"], bins=30, kde=True, color="teal")
    else:
        plt.hist(meta_faces["age"], bins=30, color="teal", alpha=0.8)
    plt.title("Distribucion de edades")
    plt.xlabel("Edad")
    plt.ylabel("Frecuencia")
    plt.show()

    # Muestra visual de imagenes
    muestra = meta_faces.sample(min(6, len(meta_faces)), random_state=SEED).reset_index(drop=True)
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))
    axes = axes.ravel()
    for i in range(6):
        axes[i].axis("off")
        if i < len(muestra):
            img = plt.imread(muestra.loc[i, "path"])
            axes[i].imshow(img, cmap="gray")
            axes[i].set_title(f"Edad: {muestra.loc[i, 'age']}")
    plt.tight_layout()
    plt.show()
else:
    print("Sin metadata de imagenes. Revisa la ruta del dataset.")

In [ ]:
# 4) Preparar matriz de entrada para baseline de regresion (solo herramientas vistas)

TARGET_SIZE = (32, 32)
MAX_IMAGES = 1200


def imagen_a_vector(path_img, target_size=(32, 32)):
    img = plt.imread(path_img)

    if img.ndim == 3:
        img = img.mean(axis=2)

    img = img.astype(np.float32)
    if img.max() > 1.0:
        img = img / 255.0

    h, w = img.shape
    th, tw = target_size

    step_h = max(1, h // th)
    step_w = max(1, w // tw)

    img_small = img[::step_h, ::step_w][:th, :tw]

    if img_small.shape[0] < th or img_small.shape[1] < tw:
        pad_h = th - img_small.shape[0]
        pad_w = tw - img_small.shape[1]
        img_small = np.pad(img_small, ((0, pad_h), (0, pad_w)), mode="edge")

    return img_small.reshape(-1)


if not meta_faces.empty:
    meta_sample = meta_faces.sample(min(MAX_IMAGES, len(meta_faces)), random_state=SEED).reset_index(drop=True)

    X_img = []
    y_img = []

    for _, row in meta_sample.iterrows():
        try:
            vec = imagen_a_vector(row["path"], target_size=TARGET_SIZE)
            X_img.append(vec)
            y_img.append(row["age"])
        except Exception:
            continue

    X_img = np.array(X_img)
    y_img = np.array(y_img)

    print(f"Matriz final baseline: X={X_img.shape}, y={y_img.shape}")
else:
    X_img = np.array([])
    y_img = np.array([])

In [ ]:
# 5) Split + entrenamiento baseline con modelos vistos (RandomForestRegressor y GradientBoostingRegressor)

if X_img.size > 0:
    X_trainval_r, X_test_r, y_trainval_r, y_test_r = train_test_split(
        X_img, y_img, test_size=0.15, random_state=SEED
    )
    X_train_r, X_val_r, y_train_r, y_val_r = train_test_split(
        X_trainval_r, y_trainval_r, test_size=0.1765, random_state=SEED
    )

    print("Shapes regresion:")
    print(f"X_train={X_train_r.shape}, X_val={X_val_r.shape}, X_test={X_test_r.shape}")

    pre_reg = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    espacios_reg = {
        "RF_Reg": (
            RandomForestRegressor(random_state=SEED),
            {
                "modelo__n_estimators": [100, 200],
                "modelo__max_depth": [None, 10, 20],
                "modelo__min_samples_split": [2, 5],
            },
        ),
        "GB_Reg": (
            GradientBoostingRegressor(random_state=SEED),
            {
                "modelo__n_estimators": [100, 200],
                "modelo__learning_rate": [0.03, 0.05, 0.1],
                "modelo__max_depth": [2, 3],
            },
        ),
    }

    resultados_reg = []
    mejores_reg = {}

    for nombre, (estimador, params) in espacios_reg.items():
        pipe_reg = Pipeline(
            steps=[
                ("preprocess", pre_reg),
                ("modelo", estimador),
            ]
        )

        gs = GridSearchCV(
            estimator=pipe_reg,
            param_grid=params,
            scoring="neg_mean_absolute_error",
            cv=3,
            n_jobs=-1,
        )
        gs.fit(X_train_r, y_train_r)

        mejor = gs.best_estimator_
        mejores_reg[nombre] = mejor

        for split, Xs, ys in [
            ("train", X_train_r, y_train_r),
            ("val", X_val_r, y_val_r),
            ("test", X_test_r, y_test_r),
        ]:
            yp = mejor.predict(Xs)
            resultados_reg.append(
                {
                    "modelo": nombre,
                    "split": split,
                    "MAE": mean_absolute_error(ys, yp),
                    "RMSE": np.sqrt(mean_squared_error(ys, yp)),
                    "R2": r2_score(ys, yp),
                }
            )

        print(f"Mejores hiperparametros ({nombre}): {gs.best_params_}")

    df_reg = pd.DataFrame(resultados_reg)
    display(df_reg.sort_values(["split", "MAE"]).reset_index(drop=True))

    mejor_reg_nombre = (
        df_reg[df_reg["split"] == "val"].sort_values("MAE", ascending=True).iloc[0]["modelo"]
    )
    print(f"Mejor baseline de regresion segun val (MAE): {mejor_reg_nombre}")

    # Reentrenamiento final con train + val
    X_train_val_reg = np.vstack([X_train_r, X_val_r])
    y_train_val_reg = np.hstack([y_train_r, y_val_r])

    modelo_reg_final = mejores_reg[mejor_reg_nombre]
    modelo_reg_final.fit(X_train_val_reg, y_train_val_reg)

    y_test_pred_reg = modelo_reg_final.predict(X_test_r)
    print("\nMetricas finales en test (baseline):")
    print(f"MAE:  {mean_absolute_error(y_test_r, y_test_pred_reg):.4f}")
    print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_test_pred_reg)):.4f}")
    print(f"R2:   {r2_score(y_test_r, y_test_pred_reg):.4f}")

    # Prueba artificial sobre una muestra real modificada
    idx = np.random.randint(0, len(X_test_r))
    muestra = X_test_r[idx].copy()
    muestra_art = np.clip(muestra * 1.10, 0, 1).reshape(1, -1)
    edad_pred_art = modelo_reg_final.predict(muestra_art)[0]

    print(f"Edad real de la muestra base: {y_test_r[idx]}")
    print(f"Edad predicha para muestra artificial: {edad_pred_art:.2f}")
else:
    print("No hay datos de imagen listos para entrenar el baseline.")

In [ ]:
# 6) CNN para regresion de edad (Keras, visto en Lecture08)

if not HAS_TF:
    print("TensorFlow/Keras no disponible. Se omite la seccion CNN.")
elif meta_faces.empty:
    print("No hay metadata de imagenes. Revisa la ruta del dataset.")
else:
    TARGET_CNN = (64, 64)
    MAX_IMAGES_CNN = min(1200, len(meta_faces))

    meta_cnn = meta_faces.sample(MAX_IMAGES_CNN, random_state=SEED).reset_index(drop=True)

    X_cnn = []
    y_cnn = []

    for _, row in meta_cnn.iterrows():
        try:
            img = keras.utils.load_img(
                row["path"], target_size=TARGET_CNN, color_mode="rgb"
            )
            arr = keras.utils.img_to_array(img).astype("float32") / 255.0
            X_cnn.append(arr)
            y_cnn.append(float(row["age"]))
        except Exception:
            continue

    X_cnn = np.array(X_cnn, dtype=np.float32)
    y_cnn = np.array(y_cnn, dtype=np.float32)

    print(f"Datos CNN: X={X_cnn.shape}, y={y_cnn.shape}")

    X_trainval_cnn, X_test_cnn, y_trainval_cnn, y_test_cnn = train_test_split(
        X_cnn, y_cnn, test_size=0.15, random_state=SEED
    )
    X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn = train_test_split(
        X_trainval_cnn, y_trainval_cnn, test_size=0.1765, random_state=SEED
    )

    print("Shapes CNN:")
    print(
        f"X_train={X_train_cnn.shape}, X_val={X_val_cnn.shape}, X_test={X_test_cnn.shape}"
    )

    def crear_cnn_regresion(input_shape):
        modelo = keras.Sequential(
            [
                keras.layers.Input(shape=input_shape),
                keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
                keras.layers.BatchNormalization(),
                keras.layers.MaxPooling2D((2, 2)),
                keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
                keras.layers.BatchNormalization(),
                keras.layers.MaxPooling2D((2, 2)),
                keras.layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
                keras.layers.MaxPooling2D((2, 2)),
                keras.layers.Flatten(),
                keras.layers.Dense(
                    128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4)
                ),
                keras.layers.Dropout(0.30),
                keras.layers.Dense(1, activation="linear"),
            ]
        )
        modelo.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss="mse",
            metrics=[
                keras.metrics.MeanAbsoluteError(name="mae"),
                keras.metrics.RootMeanSquaredError(name="rmse"),
            ],
        )
        return modelo

    cnn_model = crear_cnn_regresion(X_train_cnn.shape[1:])

    callback_es = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True
    )

    history_cnn = cnn_model.fit(
        X_train_cnn,
        y_train_cnn,
        validation_data=(X_val_cnn, y_val_cnn),
        epochs=35,
        batch_size=32,
        callbacks=[callback_es],
        verbose=0,
    )

    # Curvas train/val para detectar overfitting o underfitting
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history_cnn.history["loss"], label="train_loss")
    plt.plot(history_cnn.history["val_loss"], label="val_loss")
    plt.title("CNN regresion - perdida")
    plt.xlabel("Epoca")
    plt.ylabel("MSE")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history_cnn.history["mae"], label="train_mae")
    plt.plot(history_cnn.history["val_mae"], label="val_mae")
    plt.title("CNN regresion - MAE")
    plt.xlabel("Epoca")
    plt.ylabel("MAE")
    plt.legend()
    plt.tight_layout()
    plt.show()

    def metricas_regresion(y_true, y_pred):
        return {
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred),
            "MAPE": mean_absolute_percentage_error(y_true, y_pred),
            "MedAE": median_absolute_error(y_true, y_pred),
            "ExplainedVar": explained_variance_score(y_true, y_pred),
            "MaxError": max_error(y_true, y_pred),
        }

    filas_cnn = []
    for split, Xs, ys in [
        ("train", X_train_cnn, y_train_cnn),
        ("val", X_val_cnn, y_val_cnn),
        ("test", X_test_cnn, y_test_cnn),
    ]:
        yp = cnn_model.predict(Xs, verbose=0).ravel()
        filas_cnn.append({"modelo": "CNN_Reg", "split": split, **metricas_regresion(ys, yp)})

    df_cnn = pd.DataFrame(filas_cnn)
    display(df_cnn)

    # Reentrenamiento final en train+val y evaluacion en test
    cnn_model_final = crear_cnn_regresion(X_train_cnn.shape[1:])
    _ = cnn_model_final.fit(
        X_trainval_cnn,
        y_trainval_cnn,
        validation_split=0.1,
        epochs=35,
        batch_size=32,
        callbacks=[callback_es],
        verbose=0,
    )

    y_test_pred_cnn = cnn_model_final.predict(X_test_cnn, verbose=0).ravel()
    print("\nMetricas finales CNN en test:")
    print(metricas_regresion(y_test_cnn, y_test_pred_cnn))

    # Prueba con muestra artificial (misma forma y canales)
    idx = np.random.randint(0, len(X_test_cnn))
    img_base = X_test_cnn[idx]
    img_brillo = np.clip(img_base * 1.15, 0, 1)
    img_flip = img_base[:, ::-1, :]

    pred_base = float(cnn_model_final.predict(img_base[None, ...], verbose=0).ravel()[0])
    pred_brillo = float(cnn_model_final.predict(img_brillo[None, ...], verbose=0).ravel()[0])
    pred_flip = float(cnn_model_final.predict(img_flip[None, ...], verbose=0).ravel()[0])

    print(f"Edad real base: {y_test_cnn[idx]:.1f}")
    print(f"Prediccion imagen original: {pred_base:.2f}")
    print(f"Prediccion con mas brillo: {pred_brillo:.2f}")
    print(f"Prediccion con flip horizontal: {pred_flip:.2f}")

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_base)
    axes[0].set_title("Original")
    axes[1].imshow(img_brillo)
    axes[1].set_title("Brillo +15%")
    axes[2].imshow(img_flip)
    axes[2].set_title("Flip horizontal")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

### Cierre y puntos para sustentacion

Con las celdas anteriores ya queda resuelto el flujo pedido en `workshop2.md` para ambos problemas.

Checklist rapido para tu informe final:
1. Reportar el mejor modelo de clasificacion segun F1 en validacion y su desempeno final en test.
2. Explicar si hubo overfitting o underfitting usando la brecha train-val y las curvas.
3. En regresion, comparar baseline de sklearn vs CNN en MAE, RMSE y R2.
4. Analizar la prueba artificial (clasificacion) y la sensibilidad a cambios visuales (regresion).

Si quieres, en una siguiente iteracion te puedo dejar redactadas automaticamente las conclusiones en formato de reporte.